# 보고서 노트북 재현 검증

`verify_reproduction.ipynb`에서 만든 격리 작업 폴더를 사용합니다. 현재 분석의 추가 보고서·진단 노트북 7개를 새 커널에서 실행하고 원본 결과는 보존합니다.

In [1]:
from pathlib import Path
import json, shutil, os, time
import nbformat
from nbclient import NotebookClient
from IPython.display import display
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'environment/path_aliases.json').is_file())
ENV = ROOT/'environment'
WORK = Path(json.loads((ENV/'verification-output/summary.json').read_text(encoding='utf-8'))['workspace'])
assert WORK.is_relative_to(ENV/'validation-workspace') and WORK.is_dir()
for folder in ['jupyter-notebook','reports','documents','dataset/analysis']:
    shutil.copytree(ROOT/folder, WORK/folder, dirs_exist_ok=True)
for name in ['project_paths.ipynb','path_aliases.json','path_migration.json']:
    shutil.copy2(ENV/name, WORK/'environment'/name)
for name in ['AGENTS.md','README.md','requirements.txt']:
    shutil.copy2(ROOT/name, WORK/name)
os.environ['JUPYTER_PATH'] = str(ROOT/'.venv/share/jupyter')
os.environ['JUPYTER_PREFER_ENV_PATH'] = '1'
records = []


## 실행 결과

근거 집계 → 기준선 보고서 → 시간정보 활용 → 정밀도·노출·표본 진단 → 종합 보고서 순서입니다.

In [2]:
for path in sorted((WORK/'jupyter-notebook/20261006/report-notebooks').glob('*.ipynb')):
    if int(path.name[:2]) > 7:
        continue
    notebook = nbformat.read(path,as_version=4)
    started = time.perf_counter()
    try:
        NotebookClient(notebook,timeout=1200,kernel_name='python3',resources={'metadata':{'path':str(path.parent)}}).execute()
        records.append({'notebook':path.relative_to(WORK).as_posix(),'status':'passed','seconds':round(time.perf_counter()-started,2)})
    except Exception as exc:
        records.append({'notebook':path.relative_to(WORK).as_posix(),'status':'failed','error':str(exc)[-5000:]})
        nbformat.write(notebook,ENV/'verification-output'/('failed_'+path.name))
        (ENV/'verification-output/report_checks.json').write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')
        raise
    nbformat.write(notebook,path)
    (ENV/'verification-output/report_checks.json').write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(records))


C:\Coding\graduation-project\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()


,notebook,status,seconds
0,jupyter-notebook/20261006/report-notebooks/01_...,passed,1.85
1,jupyter-notebook/20261006/report-notebooks/02_...,passed,1.68
2,jupyter-notebook/20261006/report-notebooks/03_...,passed,1.60
3,jupyter-notebook/20261006/report-notebooks/04_...,passed,5.87
4,jupyter-notebook/20261006/report-notebooks/05_...,passed,3.56
5,jupyter-notebook/20261006/report-notebooks/06_...,passed,1.59
6,jupyter-notebook/20261006/report-notebooks/07_...,passed,1.55


## 해석

보고서와 보조 진단의 실행·경로 연결을 확인했습니다. 웹 브라우저가 없는 경우 HTML 생성기는 구조 검증 결과를 기록하며 시각 검증 통과로 표시하지 않습니다.